In [1]:
from freqtrade.configuration import Configuration
from pathlib import Path
import os
from freqtrade.data.history import load_pair_history
from freqtrade.enums import CandleType
from freqtrade.resolvers import StrategyResolver
from freqtrade.data.dataprovider import DataProvider
from plotly import graph_objects as go
from freqtrade.plot.plotting import  generate_candlestick_graph
from sklearn.cluster import KMeans
from sklearn.linear_model import LinearRegression
import numpy as np
import pandas as pd
from pandas import DataFrame
from freqtrade.strategy import IStrategy, merge_informative_pair

In [2]:
project_root = "somedir/freqtrade"
i=0
try:
    os.chdirdir(project_root)
    assert Path('docker-compose.yml').is_file()
except:
    while i<4 and (not Path('docker-compose.yml').is_file()):
        os.chdir(Path(Path.cwd(), '../'))
        i+=1
    project_root = Path.cwd()
print(Path.cwd())

/root/defi


In [3]:
def cluster_borders(dataframe):
    X = dataframe.high.values.reshape(-1,1)
    kmeans = KMeans(n_clusters=3, random_state=42).fit(X)
    dataframe['cluster'] = kmeans.predict(X)
    borders = dataframe.groupby(['cluster']).min().high.sort_values().values
    return np.append(borders, dataframe.high.max())

In [ ]:
!freqtrade download-data -c user_data/bybit_config.json --pairs $pair -t 1m

In [4]:
config = Configuration.from_files(["user_data/bybit_config.json"])
config["strategy"] = "ClusterStrategy"
data_location = config["datadir"]
pair = 'WIF/USDT:USDT'

In [5]:
candles = load_pair_history(
    datadir=data_location,
    timeframe='1m',
    pair=pair,
    data_format = "feather",
    candle_type=CandleType.FUTURES,
)
candles = candles[-240:]

In [ ]:
strategy = StrategyResolver.load_strategy(config)
strategy.dp = DataProvider(config, None, None)
strategy.ft_bot_start()
dataframe = strategy.analyze_ticker(candles, {'pair': pair})

In [76]:
dataframe = load_pair_history(
    datadir=data_location,
    timeframe='1m',
    pair=pair,
    data_format = "feather",
    candle_type=CandleType.FUTURES,
)
informative = load_pair_history(
    datadir=data_location,
    timeframe='1d',
    pair=pair,
    data_format = "feather",
    candle_type=CandleType.FUTURES,
)
informative_2 = load_pair_history(
    datadir=data_location,
    timeframe='15m',
    pair=pair,
    data_format = "feather",
    candle_type=CandleType.FUTURES,
)
dataframe = merge_informative_pair(dataframe, informative, '1m', '1d', ffill=False)

In [16]:
borders = cluster_borders(dataframe)
fig = generate_candlestick_graph(pair=pair, data=dataframe)
for border in borders:
    fig.add_hline(y=border, line_width=1, line_color='green')
fig.add_hline(y=3.5767, line_width=1, line_color='orange')
fig.add_hline(y=3.5347, line_width=1, line_color='red')
fig.add_hline(y=3.533, line_width=1, line_color='red')
fig.add_vline(x="2024-04-11 12:33:35", line_width=1, line_color="orange")
fig.update_layout(autosize=True, height=800, xaxis=dict(rangeslider=dict(visible=False)))

In [17]:
borders.tofile('user_data/notebooks/levels.csv', sep = ',')
np.genfromtxt('user_data/notebooks/levels.csv', delimiter=',')

In [ ]:
labels = [1,2,3]
colors = ['green', 'red', 'orange']
fig = generate_candlestick_graph(pair=pair, data=dataframe)

# max_values = dataframe.groupby(['label_1']).max().close.values
# min_values = dataframe.groupby(['label_1']).min().close.values
# for max_value in max_values:
#     fig.add_hline(y=max_value, line_width=2, line_color='green')
# for min_value in min_values:
#     fig.add_hline(y=min_value, line_width=2, line_color='green')

# max_values = dataframe.groupby(['label_1', 'label_2']).max().close.values
# min_values = dataframe.groupby(['label_1', 'label_2']).min().close.values
# for max_value in max_values:
#     fig.add_hline(y=max_value, line_width=1, line_color='red')
# for min_value in min_values:
#     fig.add_hline(y=min_value, line_width=1, line_color='red')

max_values = dataframe.groupby(['label_1', 'label_2', 'label_3']).max().close.values
min_values = dataframe.groupby(['label_1', 'label_2', 'label_3']).min().close.values
for max_value in max_values:
    fig.add_hline(y=max_value, line_width=0.5, line_color='orange')
for min_value in min_values:
    fig.add_hline(y=min_value, line_width=0.5, line_color='orange')

fig.update_layout(autosize=True, height=800, xaxis=dict(rangeslider=dict(visible=False)))

In [ ]:
c_min = dataframe[dataframe.cluster == dataframe.cluster.iat[-1]].close.min()
c_max = dataframe[dataframe.cluster == dataframe.cluster.iat[-1]].close.max()
timeframe = '4h'
dataframe = prepare_dataframe(timeframe)
dataframe = dataframe[(dataframe.close >= c_min) & (dataframe.close <= c_max)]
levels = pair_levels(dataframe)
fig = generate_candlestick_graph(pair=pair, data=dataframe)
for level in levels:
    fig.add_hline(y=level, line_width=1, line_color='green')
fig.update_layout(autosize=True, height=800, xaxis=dict(rangeslider=dict(visible=False)))

In [ ]:
c_min = dataframe[dataframe.cluster == dataframe.cluster.iat[-1]].close.min()
c_max = dataframe[dataframe.cluster == dataframe.cluster.iat[-1]].close.max()
timeframe = '15m'
dataframe = prepare_dataframe(timeframe)
dataframe = dataframe[(dataframe.close >= c_min) & (dataframe.close <= c_max)]
levels = pair_levels(dataframe)
fig = generate_candlestick_graph(pair=pair, data=dataframe)
for level in levels:
    fig.add_hline(y=level, line_width=1, line_color='green')
fig.update_layout(autosize=True, height=800, xaxis=dict(rangeslider=dict(visible=False)))

In [82]:
dataframe = prepare_dataframe(timeframe='1d')
X = dataframe.high.values
X = np.append(X, dataframe.low.values).reshape(-1, 1)
kmeans = KMeans(n_clusters=3, random_state=42).fit(X)
df = pd.DataFrame({'cluster':kmeans.predict(X),'values':X.flatten()})
levels = df.groupby(['cluster']).min()['values'].sort_values().values
levels = np.append(levels, df['values'].max())
dt = dataframe[dataframe.close >= levels[-2]].date.iat[0].date().strftime('%Y%m%d-')


In [ ]:
fig = generate_candlestick_graph(pair=pair, data=dataframe)
for level in levels:
    fig.add_hline(y=level, line_width=1, line_color='green')
fig.update_layout(autosize=True, height=800, xaxis=dict(rangeslider=dict(visible=False)))

In [ ]:
!freqtrade download-data -c user_data/bybit_config.json --timerange $dt --pairs $pair -t 1h

In [86]:
dataframe = prepare_dataframe(timeframe='1h')
X = dataframe.high.values
X = np.append(X, dataframe.low.values).reshape(-1, 1)
kmeans = KMeans(n_clusters=3, random_state=42).fit(X)
df = pd.DataFrame({'cluster':kmeans.predict(X),'values':X.flatten()})
levels = df.groupby(['cluster']).min()['values'].sort_values().values
levels = np.append(levels, df['values'].max())
dt = dataframe[dataframe.close >= levels[-3]].date.iat[0].date().strftime('%Y%m%d-')

In [ ]:
fig = generate_candlestick_graph(pair=pair, data=dataframe)
for level in levels:
    fig.add_hline(y=level, line_width=1, line_color='green')
fig.update_layout(autosize=True, height=800, xaxis=dict(rangeslider=dict(visible=False)))

In [ ]:
!freqtrade download-data -c user_data/bybit_config.json --timerange $20240111- --pairs $pair -t 3m

In [95]:
dataframe = prepare_dataframe(timeframe='3m')

In [96]:
def cluster(dataframe, n=3):
    X = dataframe.high.values
    X = np.append(X, dataframe.low.values).reshape(-1, 1)
    kmeans = KMeans(n_clusters=n, random_state=42).fit(X)
    return kmeans.predict(X)

In [ ]:

dataframe['label_1'] = cluster(dataframe)

grouped = dataframe.groupby(['label_1']).apply(cluster).to_dict()
for c, values in grouped.items():
    condition_1 = dataframe['label_1'] == c
    dataframe.loc[condition_1, 'label_2'] = values

grouped = dataframe.groupby(['label_1','label_2']).apply(cluster).to_dict()
for c, values in grouped.items():
    condition_1 = dataframe['label_1'] == c[0]
    condition_2 = dataframe['label_2'] == c[1]
    dataframe.loc[(condition_1 & condition_2), 'label_3'] = values

In [92]:
dataframe = prepare_dataframe(timeframe='3m')
dataframe = dataframe[(dataframe.close >= levels[-3]) & (dataframe.close <= levels[-2])]
X = dataframe.high.values
X = np.append(X, dataframe.low.values).reshape(-1, 1)
kmeans = KMeans(n_clusters=3, random_state=42).fit(X)
df = pd.DataFrame({'cluster':kmeans.predict(X),'values':X.flatten()})
levels = df.groupby(['cluster']).min()['values'].sort_values().values
levels = np.append(levels, df['values'].max())
dt = dataframe[dataframe.close >= levels[-3]].date.iat[0].date().strftime('%Y%m%d-')

In [ ]:
fig = generate_candlestick_graph(pair=pair, data=dataframe)
for level in levels:
    fig.add_hline(y=level, line_width=1, line_color='green')
fig.update_layout(autosize=True, height=800, xaxis=dict(rangeslider=dict(visible=False)))

In [66]:
dataframe = prepare_dataframe(timeframe='1d')
# levels = pair_levels(pair)
for c, level in enumerate(levels):
    dataframe.loc[(dataframe.close >= level),'cluster'] = c

In [ ]:
fig = generate_candlestick_graph(pair=pair, data=dataframe)
for level in levels:
    fig.add_hline(y=level, line_width=1, line_color='green')
fig.update_layout(autosize=True, height=800, xaxis=dict(rangeslider=dict(visible=False)))

In [46]:
timerange = '20240405-'

In [ ]:
!freqtrade download-data -c user_data/bybit_config.json --timerange $timerange --pairs $pair -t 1m

In [51]:
dataframe = load_pair_history(
        datadir=data_location,
        timeframe='1m',
        pair=pair,
        data_format = "feather",
        candle_type=CandleType.FUTURES,
    )
dataframe = dataframe[dataframe.date>='2024-04-05 02:00']

In [52]:
def pair_levels_1m(pair):
    dataframe = load_pair_history(
        datadir = config["datadir"],
        timeframe = '1m',
        pair = pair,
        data_format = "feather",
        candle_type=CandleType.FUTURES,
    )
    dataframe = dataframe[dataframe.date>='2024-04-05 02:00']
    dataframe['cluster'] = cluster(dataframe)
    levels = dataframe.groupby(['cluster']).min().close.sort_values().values
    levels = np.append(levels, dataframe.close.max())
    return levels

In [53]:
levels = pair_levels_1m(pair)
for c, level in enumerate(levels):
    dataframe.loc[(dataframe.close >= level),'cluster'] = c

In [ ]:
fig = generate_candlestick_graph(pair=pair, data=dataframe)
for level in levels:
    fig.add_hline(y=level, line_width=1, line_color='green')
fig.update_layout(autosize=True, height=800, xaxis=dict(rangeslider=dict(visible=False)))

In [ ]:
labels = [1,2,3]
colors = ['green', 'red', 'orange']
fig = generate_candlestick_graph(pair=pair, data=dataframe)

max_values = dataframe[[f'label_1', 'close']].groupby(['label_1']).max().close.values
min_values = dataframe[[f'label_1', 'close']].groupby(['label_1']).min().close.values
for max_value in max_values:
    fig.add_hline(y=max_value, line_width=2, line_color='green')
for min_value in min_values:
    fig.add_hline(y=min_value, line_width=2, line_color='green')

max_values = dataframe[[f'label_1', 'label_2', 'close']].groupby(['label_1', 'label_2']).max().close.values
min_values = dataframe[[f'label_1', 'label_2', 'close']].groupby(['label_1', 'label_2']).min().close.values
for max_value in max_values:
    fig.add_hline(y=max_value, line_width=1, line_color='red')
for min_value in min_values:
    fig.add_hline(y=min_value, line_width=1, line_color='red')

max_values = dataframe[[f'label_1', 'label_2', 'label_3', 'close']].groupby(['label_1', 'label_2', 'label_3']).max().close.values
min_values = dataframe[[f'label_1', 'label_2', 'label_3', 'close']].groupby(['label_1', 'label_2', 'label_3']).min().close.values
for max_value in max_values:
    fig.add_hline(y=max_value, line_width=0.5, line_color='orange')
for min_value in min_values:
    fig.add_hline(y=min_value, line_width=0.5, line_color='orange')

fig.update_layout(autosize=True, height=800, xaxis=dict(rangeslider=dict(visible=False)))

In [10]:
from freqtrade.data.btanalysis import load_trades_from_db
trades = load_trades_from_db("sqlite:///user_data/cluster_strategy_dry.sqlite")

In [20]:
trades.head(1)

,pair,stake_amount,max_stake_amount,amount,open_date,close_date,open_rate,close_rate,fee_open,fee_close,...,stop_loss_ratio,min_rate,max_rate,is_open,enter_tag,leverage,is_short,open_timestamp,close_timestamp,orders
0,GMT/USDT:USDT,49.1905,49.1905,131.0,2024-03-29 05:30:04+00:00,2024-03-29 09:04:01+00:00,0.3755,0.3803,0.0001,0.0001,...,-0.036879,0.3667,0.3803,False,,1.0,True,1711690204667,1.711703e+12,"[{'amount': 131.0, 'safe_price': 0.3755, 'ft_o..."


In [ ]:
def populate_indicators(self, dataframe: DataFrame, metadata: dict) -> DataFrame:

        dataframe['label_1'] = self.cluster(dataframe)

        grouped = dataframe.groupby(['label_1']).apply(self.cluster).to_dict()
        for c, values in grouped.items():
            condition_1 = dataframe['label_1'] == c
            dataframe.loc[condition_1, 'label_2'] = values

        grouped = dataframe.groupby(['label_1','label_2']).apply(self.cluster).to_dict()
        for c, values in grouped.items():
            condition_1 = dataframe['label_1'] == c[0]
            condition_2 = dataframe['label_2'] == c[1]
            dataframe.loc[(condition_1 & condition_2), 'label_3'] = values

        return dataframe

def populate_entry_trend(self, dataframe: DataFrame, metadata: dict) -> DataFrame:

    dataframe.loc[
        (
            (dataframe.label_3.shift(1) == 0) &
            (dataframe.label_3 == 1)
        ),
        'enter_long'
    ] = 1

    dataframe.loc[
        (
            (dataframe.label_3.shift(1) == 2) &
            (dataframe.label_3 == 1)
        ),
        'enter_short'
    ] = 1

    return dataframe

def populate_exit_trend(self, dataframe: DataFrame, metadata: dict) -> DataFrame:

    return dataframe

def position_size(self, total_asset, risk):

    size = total_asset * self.total_risk / risk if risk > self.total_risk else total_asset

    return size

def custom_stake_amount(self, pair: str, current_time: datetime, current_rate: float,
                        proposed_stake: float, min_stake: Optional[float], max_stake: float,
                        leverage: float, entry_tag: Optional[str], side: str,
                        **kwargs) -> float:

    dataframe, _ = self.dp.get_analyzed_dataframe(pair=pair, timeframe=self.timeframe)
    mins = dataframe.groupby([f'label_{i}' for i in [1,2,3]]).min().close.sort_values().values
    prev_close = dataframe.close.iat[-2]
    
    if side == 'long':
        risk = 1 - mins[mins < prev_close][-2] / prev_close
    else:
        risk = mins[mins > prev_close][1] / prev_close - 1
    
    stake = self.position_size(max_stake, risk)

    return stake

def custom_stoploss(self, pair: str, trade: 'Trade', current_time: datetime,
                    current_rate: float, current_profit: float, after_fill: bool, 
                    **kwargs) -> Optional[float]:

    dataframe, _ = self.dp.get_analyzed_dataframe(pair, self.timeframe)
    mins = dataframe.groupby([f'label_{i}' for i in [1,2,3]]).min().close.sort_values().values
    prev_close = dataframe.close.iat[-2]

    if trade.is_short:
        return stoploss_from_absolute(
            mins[mins > prev_close][1],
            prev_close,
            is_short=trade.is_short,
            leverage=trade.leverage
        )
    
    return stoploss_from_absolute(
            mins[mins < prev_close][-2],
            prev_close,
            is_short=trade.is_short,
            leverage=trade.leverage
        )

In [ ]:
class ClusterStrategy(IStrategy):

    INTERFACE_VERSION = 3

    can_short: bool = True

    stoploss = -0.1

    timeframe = '1m'

    total_risk = 0.01

    process_only_new_candles = True

    use_exit_signal = False

    exit_profit_only = False

    ignore_roi_if_entry_signal = False

    use_custom_stoploss = True

    position_adjustment_enable = False

    startup_candle_count: int = 300

    order_types = {
        'entry': 'limit',
        'exit': 'limit',
        'stoploss': 'limit',
        'stoploss_on_exchange': True
    }

    order_time_in_force = {
        'entry': 'GTC',
        'exit': 'GTC'
    }

    @property
    def protections(self):
        return [
            {
                "method": "StoplossGuard",
                "lookback_period_candles": 24,
                "trade_limit": 2,
                "stop_duration_candles": 4,
                "required_profit": 0.0,
                "only_per_pair": True,
                "only_per_side": False
            }
        ]

    def cluster(self, dataframe):
        X = dataframe['close'].values.reshape(-1,1)
        kmeans = KMeans(n_clusters=5, random_state=42).fit(X)
        return kmeans.predict(X)

    def pair_levels(self, pair):
        dataframe = load_pair_history(
            datadir = self.config["datadir"],
            timeframe = self.timeframe,
            pair = pair,
            data_format = "feather",
            candle_type=CandleType.FUTURES,
        )
        # dataframe = dataframe[dataframe.date>='2024-04-05 02:00']
        dataframe['cluster'] = self.cluster(dataframe)
        levels = dataframe.groupby(['cluster']).min().close.sort_values().values
        levels = np.append(levels, dataframe.close.max())
        return levels

    def populate_indicators(self, dataframe: DataFrame, metadata: dict) -> DataFrame:

        levels = self.pair_levels(metadata['pair'])
        for c, level in enumerate(levels):
            dataframe.loc[(dataframe.close >= level),'cluster'] = c

        return dataframe

    def populate_entry_trend(self, dataframe: DataFrame, metadata: dict) -> DataFrame:

        dataframe.loc[
            (
                (dataframe.cluster.shift(1) < dataframe.cluster)
            ),
            'enter_long'
        ] = 1

        dataframe.loc[
            (
                (dataframe.cluster.shift(1) > dataframe.cluster)
            ),
            'enter_short'
        ] = 1

        return dataframe

    def populate_exit_trend(self, dataframe: DataFrame, metadata: dict) -> DataFrame:

        return dataframe

    def position_size(self, total_asset, risk):

        size = total_asset * self.total_risk / risk if risk > self.total_risk else total_asset

        return size

    def custom_stake_amount(self, pair: str, current_time: datetime, current_rate: float,
                            proposed_stake: float, min_stake: Optional[float], max_stake: float,
                            leverage: float, entry_tag: Optional[str], side: str,
                            **kwargs) -> float:

        dataframe, _ = self.dp.get_analyzed_dataframe(pair=pair, timeframe=self.timeframe)
        prev_close = dataframe.close.iat[-2]
        levels = self.pair_levels(pair)
        
        if side == 'long':
            levels = np.append(levels, dataframe.low.iat[-2])
            levels = np.sort(levels)
            risk = 1 - levels[levels < prev_close][-2] / prev_close
        else:
            levels = np.append(levels, dataframe.high.iat[-2])
            levels = np.sort(levels)
            risk = levels[levels > prev_close][1] / prev_close - 1
        
        stake = self.position_size(max_stake, risk)

        return stake

    def custom_stoploss(self, pair: str, trade: 'Trade', current_time: datetime,
                        current_rate: float, current_profit: float, after_fill: bool, 
                        **kwargs) -> Optional[float]:

        dataframe, _ = self.dp.get_analyzed_dataframe(pair, self.timeframe)
        prev_close = dataframe.close.iat[-2]
        levels = self.pair_levels(pair)
        stop = trade.get_custom_data(key='stop')

        if stop:
            if trade.is_short:
                levels = np.append(levels, stop)
                levels = np.sort(levels)
                return stoploss_from_absolute(
                    levels[levels > prev_close][1],
                    prev_close,
                    is_short=trade.is_short,
                    leverage=trade.leverage
                )
            levels = np.append(levels, stop)
            levels = np.sort(levels)
            return stoploss_from_absolute(
                    levels[levels < prev_close][-2],
                    prev_close,
                    is_short=trade.is_short,
                    leverage=trade.leverage
                )
        
        return -1

    def order_filled(self, pair: str, trade: Trade, order: 'Order', current_time: datetime, **kwargs) -> None:

        dataframe, _ = self.dp.get_analyzed_dataframe(trade.pair, self.timeframe)
        prev_candle = dataframe.iloc[-2].squeeze()

        if trade.nr_of_successful_entries == 1:
            if trade.is_short:
                trade.set_custom_data(key='stop', value=prev_candle['high'])
            else:
                trade.set_custom_data(key='stop', value=prev_candle['low'])

        return None
    
    def bot_loop_start(self, current_time: datetime, **kwargs) -> None:

        pairs = self.dp.current_whitelist()

        if self.config['runmode'].value in ('live'):
            if self.wallets:
                for pair in pairs:
                    ticker = self.dp.ticker(pair)
                    self.dp.send_msg(self.wallets.get_total(ticker))
                self.dp.send_msg(self.wallets.get_total('USDT'))
        
        for pair in pairs:
            dataframe, _ = self.dp.get_analyzed_dataframe(pair, self.timeframe)
            if not dataframe.empty:
                prev_close = dataframe.close.iat[-2]
                levels = self.pair_levels(pair)
                if prev_close > levels[-1]:
                    self.dp.send_msg('Price crossed above cluster')
                
                if prev_close < levels[0]:
                    self.dp.send_msg('Price crossed below cluster')

In [ ]:
def pair_levels(self, pair):
    
    # dataframe = load_pair_history(
    #     datadir = self.config["datadir"],
    #     timeframe = self.timeframe,
    #     pair = pair,
    #     data_format = "feather",
    #     candle_type=CandleType.FUTURES,
    # )

    # dataframe['label_1'] = self.cluster(dataframe)

    # grouped = dataframe.groupby(['label_1']).apply(self.cluster).to_dict()
    # for c, values in grouped.items():
    #     condition_1 = dataframe['label_1'] == c
    #     dataframe.loc[condition_1, 'label_2'] = values

    # grouped = dataframe.groupby(['label_1','label_2']).apply(self.cluster).to_dict()
    # for c, values in grouped.items():
    #     condition_1 = dataframe['label_1'] == c[0]
    #     condition_2 = dataframe['label_2'] == c[1]
    #     dataframe.loc[(condition_1 & condition_2), 'label_3'] = values

    # grouped = dataframe.groupby(['label_1','label_2','label_3']).apply(self.cluster).to_dict()
    # for c, values in grouped.items():
    #     condition_1 = dataframe['label_1'] == c[0]
    #     condition_2 = dataframe['label_2'] == c[1]
    #     condition_3 = dataframe['label_3'] == c[2]
    #     dataframe.loc[(condition_1 & condition_2 & condition_3), 'label_4'] = values

    # levels = dataframe.groupby([f'label_{i}' for i in [1,2,3,4]]).min().close.sort_values().values
    # levels = np.append(levels, dataframe.close.max())

    levels = np.genfromtxt('user_data/notebooks/levels.csv', delimiter=',')

    return levels

In [ ]:
from freqtrade.persistence import Trade
# ...
open_trades = Trade.get_open_trade_count()
open_trades

In [1]:
import pandas as pd
import numpy as np
import sqlite3

In [4]:
con = sqlite3.connect("../cluster_strategy_dry.sqlite")

In [14]:
tables = pd.read_sql_query("SELECT name FROM sqlite_master WHERE type='table';", con)

In [19]:
trade_custom_data = pd.read_sql_query("SELECT * FROM trade_custom_data", con)
trade_custom_data.head()

,id,ft_trade_id,cd_key,cd_type,cd_value,created_at,updated_at
0,1,1,stop,float64,3.6866,2024-04-11 10:17:06.275992,2024-04-11 10:34:36.269336
1,2,1,OB,dict,"{""symbol"": ""WIF/USDT:USDT"", ""bids"": [[3.6827, ...",2024-04-11 10:34:36.473608,None
2,3,2,stop,float64,3.5642,2024-04-11 11:52:06.263722,2024-04-11 12:33:30.194038
3,4,2,OB,dict,"{""symbol"": ""WIF/USDT:USDT"", ""bids"": [[3.5745, ...",2024-04-11 11:52:06.795589,2024-04-11 12:33:30.402535
4,5,3,stop,float64,3.5341,2024-04-11 12:34:00.192018,2024-04-11 13:44:11.585838


In [ ]:
value = trade_custom_data.cd_value[1]
value = value.replace('null','None')

In [38]:
pd.DataFrame(eval(value))

,symbol,bids,asks,timestamp,datetime,nonce
0,WIF/USDT:USDT,"[3.6827, 405.0]","[3.6835, 167.0]",1712831676342,2024-04-11T10:34:36.342Z,None
1,WIF/USDT:USDT,"[3.6825, 37.0]","[3.6837, 62.0]",1712831676342,2024-04-11T10:34:36.342Z,None
2,WIF/USDT:USDT,"[3.6824, 85.0]","[3.6839, 54.0]",1712831676342,2024-04-11T10:34:36.342Z,None
3,WIF/USDT:USDT,"[3.6822, 19.0]","[3.6841, 19.0]",1712831676342,2024-04-11T10:34:36.342Z,None
4,WIF/USDT:USDT,"[3.6821, 19.0]","[3.6842, 37.0]",1712831676342,2024-04-11T10:34:36.342Z,None
...,...,...,...,...,...,...
195,WIF/USDT:USDT,"[3.6616, 323.0]","[3.7124, 14.0]",1712831676342,2024-04-11T10:34:36.342Z,None
196,WIF/USDT:USDT,"[3.6615, 10.0]","[3.7126, 8630.0]",1712831676342,2024-04-11T10:34:36.342Z,None
197,WIF/USDT:USDT,"[3.6614, 1.0]","[3.7128, 49.0]",1712831676342,2024-04-11T10:34:36.342Z,None
198,WIF/USDT:USDT,"[3.6613, 52.0]","[3.7132, 5.0]",1712831676342,2024-04-11T10:34:36.342Z,None
